[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sitcv/cv-course5/blob/main/lanes-detection.ipynb)

In [5]:
# lines.mp4 を data/lines.mp4 にダウンロード

import os
import urllib.request

BASE = "https://raw.githubusercontent.com/sitcv/cv-course5/main/data/"
SAVE_DIR = "data"

os.makedirs(SAVE_DIR, exist_ok=True)

for name in ["lines.mp4"]:
    filepath = os.path.join(SAVE_DIR, name)
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(BASE + name, filepath)
        print("downloaded", filepath)
    else:
        print("already exists:", filepath)

already exists: data\lines.mp4


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

#画像からエッジを検出する
def canny(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    canny = cv2.Canny(blur, 50, 150)
    return canny

#平均ラインの座標を計算する
"""
This sets y1 to the height of the image (the bottom of the image), 
meaning that y1 corresponds to the bottom of the image where the line will start.
This sets y2 to be at 60% of the image height (from the bottom). 
The idea is that lane lines usually extend from the bottom of the image to somewhere higher up, 
which represents a reasonable region for detecting the lanes in a typical road scenario.
"""
def make_coordinates(image, line_parameters):
    try:
        slope, intercept = line_parameters
        # print(image.shape)
        y1 = image.shape[0]
        y2 = int(y1 * (1 / 2))
        x1 = int((y1 - intercept) / slope)
        x2 = int((y2 - intercept) / slope)
    except Exception  as e:
        print(e)
        y1=0;y2=0;x1=0;x2=0
    
    return np.array([x1, y1, x2, y2])

#レインの左(右)ラインの傾きと切片の平均を計算して、その平均から左右ラインを計算する
def average_slope_intercept(image, lines):
    left_fit = []
    right_fit = []
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        parameters = np.polyfit((x1, x2), (y1, y2), 1)
        slope = parameters[0]
        intercept = parameters[1]
        if slope < 0:
            left_fit.append((slope, intercept))
        else:
            right_fit.append((slope, intercept))
    left_fit_average = np.average(left_fit, axis=0)
    right_fit_average = np.average(right_fit, axis=0)
    left_line = make_coordinates(image, left_fit_average)
    right_line = make_coordinates(image, right_fit_average)
    return np.array([left_line, right_line])

#検出されたレインの画像を作成する
def display_lines(image, lines):
    line_image = np.zeros_like(image)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line.reshape(4)
            # or x1, y1, x2, y2 = line
            cv2.line(line_image, (x1, y1), (x2, y2), (255, 0, 0), 10)
    return line_image

#車のはしているレインの部分だけにフォーカスして、レインを検出する
def region_of_intrest(image):
    height, weight = image.shape
    polygons = np.array([[(weight//5, height), (weight*3//4, height), (weight//2, height//3)]])
    #polygons = np.array([[(200, height), (1100, height), (550, 250)]])
    mask = np.zeros_like(image)
    cv2.fillPoly(mask, polygons, 255)
    masked_image = cv2.bitwise_and(image, mask)  #
    return masked_image


cap = cv2.VideoCapture("data/lines.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    
    if ret:
        canny_image = canny(frame)
        cropped_image = region_of_intrest(canny_image)
        """ """
        lines = cv2.HoughLinesP(
            cropped_image, 2, 2*np.pi/180, 100, np.array([]), minLineLength=40, maxLineGap=5
        )
        
        if lines.size > 0:
            averaged_lines = average_slope_intercept(frame, lines)
            line_image = display_lines(frame, averaged_lines)
            combo_image = cv2.addWeighted(frame, 0.8, line_image, 1, 1)      
            cv2.imshow("result", combo_image)
            
        #cv2.imshow("result", line_image)
        
    if cv2.waitKey(1) == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

c:\Users\lixinxiao\sitcv\cv-course5\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:552: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
c:\Users\lixinxiao\sitcv\cv-course5\.venv\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64 object
cannot unpack non-iterable numpy.float64